In [1]:
!pip install --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached numpy-2.4.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/124.9 MB 5.2 MB/s eta 0:00:24
   - -------------------------------------- 4.5/124.9 MB 14.3 MB/s eta 0:00:09
   -- ------------------------------------- 7.9/124.9 MB 16.3 MB/s eta 0:00:08
   ---- ----------------------------------- 14.4/124.9 MB 19.8 MB/s eta 0:00:06
   ------ --------------------------------- 21.0/124.9 MB 21.7 MB/s eta 0:00:05
   -------- ------------------------------- 27.0/124.9 MB 23.0 MB/s eta 0:00:05
   -

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\PGCP-AI\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
###-----------------
### Import Libraries
###-----------------

from pathlib import Path  # Import Path for file system path operations and management
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split  # for train-test splitting
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix,f1_score,ConfusionMatrixDisplay

import tensorflow as tf
import torch
import torch.nn as nn
from torchsummary import summary
from collections.abc import Callable    # type hinting callable objects/functions
from typing import Literal      # literal type hints to restrict variable values

from torch.utils.data import Dataset, DataLoader


from torchvision import transforms, datasets
from torchvision.transforms import v2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2





C:\Users\PGCP-AI\AppData\Local\anaconda3\envs\dnn\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


ModuleNotFoundError: No module named 'albumentations'

In [3]:
###----------------------
### Some basic parameters
###----------------------

inpDir = Path('..') / '..' / 'input'
outDir = Path('..') / 'output'
modelDir = Path('..') / 'models'
subDir = 'flower_photos'
altName = 'cnn_base'

RANDOM_STATE = 24 # for initialization ----- REMEMBER: to remove at the time of promotion to production
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
# rng = np.random.default_rng(seed = RANDOM_STATE) # Set Random Seed for reproducible  results

EPOCHS = 200 # number of epochs
NO_SAVE_EPOCHS=30

BATCH_SIZE = 16             # <------- Look at TRAIN_SIZE too
ALPHA = 0.001 # learning rate
WEIGHT_DECAY=1e-5
EPSILON=1e-8
TRAIN_SIZE=184*BATCH_SIZE   #(3670*0.8)//BATCH_SIZE*BATCH_SIZE
# TEST_SIZE = 0.2

IMG_HEIGHT=224  #?????
IMG_WIDTH=224  #Need to change for each model
# parameters for Matplotlib
params = {'legend.fontsize': 'medium',
          'figure.figsize': (15, 6),
          'axes.labelsize': 'medium',
          'axes.titlesize':'x-large',
          'xtick.labelsize':'medium',
          'ytick.labelsize':'medium'
         }

plt.rcParams.update(params)

CMAP = plt.cm.coolwarm
plt.style.use('seaborn-v0_8-darkgrid') # plt.style.use('ggplot')

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import matplotlib. pyplot as plt
from sklearn.metrics import  confusion_matrix, ConfusionMatrixDisplay, f1_score
from scipy.interpolate import make_interp_spline

def fn_plot_torch_hist(hist_df: pd.DataFrame):
    """
    Plots the training and validation loss and accuracy curves from a PyTorch training history DataFrame.

    Args:
        hist_df (pd.DataFrame): A pandas DataFrame with five columns:
                                - First column: epoch (x-axis values)
                                - Second & third columns: losses (train & validation)
                                - Fourth & fifth columns: accuracies (train & validation)

    Returns:
        None: Displays the matplotlib plots.
    """
    # Ensure the DataFrame has exactly five columns
    if hist_df.shape[1] < 5:
        raise ValueError("The DataFrame must have atleast five columns: epoch, train_loss, val_loss, train_acc, val_acc.")

    # Extract column names for better readability and maintainability
    #epoch loss	           test_loss	 acc	        test_acc
    x_col, train_loss_col, val_loss_col, train_acc_col, val_acc_col, *cols= hist_df.columns

    # Instantiate figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    facecolor = 'cyan'  # Text box background color
    fontsize = 12  # Font size for annotations
    props = dict(boxstyle='round', facecolor=facecolor, alpha=0.5)  # Text box properties

    # First subplot: Loss curves
    ax = axes[0]
    hist_df.plot(x=x_col, y=[train_loss_col, val_loss_col], ax=ax)

    # Annotation: Final losses
    final_losses_text = f"Loss:\nTrain: {hist_df[train_loss_col].iloc[-1]:.4f}\nVal: {hist_df[val_loss_col].iloc[-1]:.4f}"
    ax.text(0.3, 0.95, final_losses_text, transform=ax.transAxes, fontsize=fontsize,
            verticalalignment='top', bbox=props)

    # Annotation: Minimum validation loss
    best_val_loss_idx = hist_df[val_loss_col].idxmin()
    best_val_loss_epoch = hist_df.loc[best_val_loss_idx, x_col]
    best_val_loss_value = hist_df.loc[best_val_loss_idx, val_loss_col]
    ax.annotate(f"Min: {best_val_loss_value:.4f}",
                xy=(best_val_loss_epoch, best_val_loss_value),
                xytext=(best_val_loss_epoch - 2, best_val_loss_value + 0.05),
                fontsize=fontsize, ha='right', va='bottom', bbox=props,
                arrowprops=dict(facecolor=facecolor, shrink=0.05))
    ax.axvline(x=best_val_loss_epoch, color='green', linestyle='-.', lw=2)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("Loss Curves")
    ax.legend(loc='upper left')
    ax.grid(True)

    # Second subplot: Accuracy curves
    ax = axes[1]
    hist_df.plot(x=x_col, y=[train_acc_col, val_acc_col], ax=ax)

    # Annotation: Final accuracies
    final_acc_text = f"Accuracy:\nTrain: {hist_df[train_acc_col].iloc[-1]:.4f}\nVal: {hist_df[val_acc_col].iloc[-1]:.4f}"
    ax.text(0.3, 0.2, final_acc_text, transform=ax.transAxes, fontsize=fontsize,
            verticalalignment='top', bbox=props)

    # Annotation: corresponding validation accuracy
    best_val_acc_epoch = hist_df.loc[best_val_loss_idx, x_col]
    best_val_acc_value = hist_df.loc[best_val_loss_idx, val_acc_col]
    ax.annotate(f"Max: {best_val_acc_value:.4f}",
                xy=(best_val_acc_epoch, best_val_acc_value),
                xytext=(best_val_acc_epoch - 2, best_val_acc_value - 0.05),
                fontsize=fontsize, ha='right', va='top', bbox=props,
                arrowprops=dict(facecolor=facecolor, shrink=0.05))
    ax.axvline(x=best_val_acc_epoch, color='green', linestyle='-.', lw=2)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.set_title("Accuracy Curves")
    ax.legend(loc='lower left')
    ax.grid(True)

    plt.tight_layout()
    plt.show()


def fn_plot_confusion_matrix(y_true, y_pred, labels):
    '''
    Args:
        y_true: Ground Truth
        y_pred : Predictions
        labels : dictionary
                  {0: 'Goal Keeper',
                  1: 'Defender',
                  2: 'Mid-Fielder',
                  3: 'Forward'}

    '''

    cm  = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=labels.values())

    fig, ax = plt.subplots(figsize = (4,4))

    disp.plot(ax = ax, cmap = 'Blues', xticks_rotation = 'vertical', colorbar=False)

    # Disable the grid
    ax.grid(False)

    title_str = f'F1 Score : {f1_score(y_true, y_pred, average='weighted'):0.5f}'
    ax.set_title(title_str)

    plt.show()

In [ ]:
# Check if all directories are present
# outDir.mkdir(parents=True, exist_ok=True)

#modelSubDir = modelDir/ subDir
#modelSubDir.mkdir(parents=True, exist_ok=True)

In [5]:
import pathlib
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

data_dir = tf.keras.utils.get_file(origin=dataset_url,
                                   fname='flower_photos',
                                   untar=True)
data_dir = pathlib.Path(data_dir)/subDir


# data_dir ='./flower_photos/'
data_dir

WindowsPath('C:/Users/PGCP-AI/.keras/datasets/flower_photos/flower_photos')

In [ ]:
# List only directories
directories = [item for item in data_dir.iterdir() if item.is_dir()]
directories

In [ ]:
def fn_imshow(img, ax=None, title=None, normalize=True):
    """Imshow for Tensor."""

    if ax is None:
        fig, ax = plt.subplots()

    img = img.numpy().transpose((1, 2, 0))
    if normalize:
        mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
        std = np.array([0.229, 0.224, 0.225])   # ImageNet std

        img = std * img + mean    # Unnormalize the image
        img = np.clip(img, 0, 1)  # Clip to [0, 1] range

    ax.imshow(img)

    if title is not None:
        plt.title(title)

    ax.axis("off")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.tick_params(axis='both', which="both", length=0)
    ax.grid(False)

    ax.set_xticklabels('')
    ax.set_yticklabels('')

    return ax

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# #Tranformers
# class DataTransformer:
#     def __init__(self):
#         self.scaler=StandardScaler()
#         self.encoder=LabelEncoder()
#         self.is_fitted=False

#     def fit(self,X,y):
#         self.scaler.fit(X,y)
#         self.encoder.fit(y)
#         self.is_fitted=True

#     def transform(self,X,y):
#         if not self.is_fitted:
#             raise ValueError('Transformers not ready yet')

#         return self.scaler.transform(X), self.encoder(y)

#      def fit_transform(self,X,y):
#         self.fit(X,y)

#         return self.scaler.transform(X), self.encoder(y)

In [ ]:
class_names= {0:'daisy', 1:'dandelion', 2:'roses', 3:'sunflowers',4: 'tulips'}


### dataset

In [ ]:

from math import e
#every time we request it will give batch of random 32 rows from dataset
class FlowerDataset(Dataset):

    def __init__(self,data_dir,transform=None):

        super(FlowerDataset,self).__init__()

        self.data_dir=data_dir
        self.transform= transform

        self.samples=[]
        self.class_to_idx={}
        self._build_dataset()

    def _build_dataset(self):
      class_dirs = [d for d in self.data_dir.iterdir() if d.is_dir()]
      self.classes = sorted([d.name for d in class_dirs])
      self.class_to_idx = {c: i for i, c in enumerate(self.classes)}

      for class_dir in class_dirs:
        class_idx = self.class_to_idx[class_dir.name]
        for img_path in class_dir.glob('*.*'):
          if img_path.is_file() and img_path.suffix in ['.jpg', '.jpeg', '.png', 'bmp', 'gif','tiff']:
            self.samples.append((img_path, class_idx))


    def __len__(self):
        return len(self.samples)

    def __getitem__(self,idx):
        img_path, label= self.samples[idx]
        try:

          img=cv2.imread(str(img_path), cv2.IMREAD_COLOR)

          if img is None:
              raise ValueError(f'Could not read image: {img_path}')

          img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

          if self.transform:
            image = self.transform(image=img)['image']
          return image,label

        except Exception as e:
          print(f'Error loading image {img_path}')
          dummy_image = torch.zeros((3, IMG_HEIGHT, IMG_WIDTH))
        return dummy_image, label




In [ ]:
def get_transforms(img_height=IMG_HEIGHT, img_width=IMG_WIDTH):
    train_transform = A.Compose([
        A.Resize(width=img_width, height=img_height, interpolation=cv2.INTER_LANCZOS4),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225],
                    max_pixel_value=255.0),
        ToTensorV2()
    ])

    test_transform = A.Compose([
        A.Resize(width=img_width, height=img_height, interpolation=cv2.INTER_LANCZOS4),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225],
                     max_pixel_value=255.0),
        ToTensorV2()
    ])

    return train_transform, test_transform

In [ ]:
# Aug 3: noise transformations
        A.OneOf([
            #A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.GaussNoise(std_range=(0.1, 0.2), p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=0.5),
            A.MotionBlur(blur_limit=(3, 5), p=0.3),], p=0.7),

        # Aug 4: Cutouts & Occlusions
        A.CoarseDropout(num_holes_range=(3, 6),
                        hole_height_range=(img_height//20, img_height//10),
                        hole_width_range=(img_width//20, img_width//10),
                        fill="random_uniform",
                        p=0.2),

        # Aug 5: sharpening & blurring
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.8, 1.2), p=0.2),
        
        # Aug 6: Normalization & ToTensor

In [ ]:
train_transform, test_transform = get_transforms(IMG_HEIGHT,IMG_WIDTH)




In [ ]:
def create_datasets(data_dir,
                    train_transform= train_transform,
                    test_transform = test_transform):
  train_dataset = FlowerDataset(data_dir, transform=train_transform)
  test_dataset = FlowerDataset(data_dir, transform=test_transform)

  indices = torch.randperm(len(train_dataset)).tolist()
  dataset_train = torch.utils.data.Subset(train_dataset, indices[:TRAIN_SIZE])
  dataset_test = torch.utils.data.Subset(test_dataset, indices[TRAIN_SIZE:])

  print(f'Total samples: {len(train_dataset)}')
  print(f'Train size: {len(dataset_train)}')
  print(f'Test size: {len(dataset_test)}')
  print(f'Classes: {train_dataset.classes}')
  print(f'Classes mapping: {train_dataset.class_to_idx}')

  return dataset_train, dataset_test



In [ ]:
train_dataset, test_dataset = create_datasets(data_dir, train_transform, test_transform)

In [ ]:
train_loader= DataLoader(train_dataset, batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=4)
test_loader= DataLoader(test_dataset, batch_size=BATCH_SIZE,
                        shuffle=False,num_workers=4)

In [ ]:
images,labels= next(iter(train_loader))

cols=8
rows= BATCH_SIZE//8

fig,axes = plt.subplots(rows,cols,figsize = (16 , rows * 2))
fig.subplots_adjust(left = 0.0, right = 1.0, bottom = 0.0, top = 1.0,
                    hspace = 0.05, wspace = 0.05)

axes = axes.ravel()

for i in range(BATCH_SIZE):
  ax=axes[i]
  fn_imshow(images[i], ax=ax, title=class_names[labels[i].item()], normalize = True)

plt.tight_layout()
plt.show()

# Defne model

In [ ]:
class FlowerModel(nn.Module):

    def __init__(self,
                 numChannels , classes):

        super(FlowerModel, self).__init__()

        dor1 = 0.1
        dor2 = 0.2
        dor3 = 0.3
        dor4 = 0.4
        dor5 = 0.5
        dor6 = 0.6
        dor7 = 0.6
        # dor8 = 0.8
        # dor9 = 0.9

  #set1
        # in_channels1 = 1
        out_channels1 = 64
        self.conv1 = nn.Conv2d(in_channels = numChannels,
                              out_channels = out_channels1,
                              kernel_size = 5)#(220 x 220 x 64)
        # self.bn1 = nn.BatchNorm2d(out_channels1)
        self.activ1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size = (2,2), stride = (2,2))#110 x 110 x 128
        # self.do1 = nn.Dropout(p = dor1)


  #set 2
        out_channels2 = 128
        self.conv2 = nn.Conv2d(in_channels = out_channels1,
                              out_channels = out_channels2,
                              kernel_size = 3)#108 x 108 x 128
        # self.bn2 = nn.BatchNorm2d(out_channels2)
        self.activ2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool2d(kernel_size = (2,2), stride = (2,2))#54 x 54 x 256
        # self.do2 = nn.Dropout(p = dor2)


 #set 3
        out_channels3 = 256
        self.conv3 = nn.Conv2d(in_channels = out_channels2,
                              out_channels = out_channels3,
                              kernel_size = 3)# 52 x 52 x 256
        # self.bn1 = nn.BatchNorm2d(out_channels3)
        self.activ3 = nn.ReLU()
        self.maxpool3 = nn.MaxPool2d(kernel_size = (2,2), stride = (2,2))#26 x 26 x 512
        # self.do1 = nn.Dropout(p = dor3)

  #set 4
        out_channels4 = 512
        self.conv4 = nn.Conv2d(in_channels = out_channels3,
                              out_channels = out_channels4,
                              kernel_size = 3)# 24 x 24 x 512
        # self.bn1 = nn.BatchNorm2d(out_channels3)
        self.activ4 = nn.ReLU()
        self.maxpool4= nn.MaxPool2d(kernel_size = (2,2), stride = (2,2))#12 x 12 x 1024
        # self.do1 = nn.Dropout(p = dor4)


  #set 5
        out_channels5 = 1024
        self.conv5 = nn.Conv2d(in_channels = out_channels4,
                              out_channels = out_channels5,
                              kernel_size = 3)# 10 x 10 x 1024
        # self.bn1 = nn.BatchNorm2d(out_channels3)
        self.activ5 = nn.ReLU()
        self.maxpool5 = nn.MaxPool2d(kernel_size = (2,2), stride = (2,2))#5 x 5 x 2048
        # self.do1 = nn.Dropout(p = dor5)


  #set6
        out_channels6 = 2048
        self.conv6 = nn.Conv2d(in_channels = out_channels5,
                              out_channels = out_channels6,
                              kernel_size = 3)# 3 x 3x 2048
        # self.bn1 = nn.BatchNorm2d(out_channels3)
        self.activ6 = nn.ReLU()
        # self.do1 = nn.Dropout(p = dor6)

  #set7
        self.fc_input_size = out_channels6 * 3 * 3
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1,1)) # output_shape = (1 x 1 x 2048)


        self.fc1 = nn.Linear(self.fc_input_size, 512)
        # self.bn4 = nn.BatchNorm1d(128)
        self.activ7 = nn.ReLU()
        # self.do4 = nn.Dropout(dor7)

        self.fc2 = nn.Linear(512, classes)

    def forward(self, x):

        # x = self.do1(self.maxpool1(self.activ1(self.bn1(self.conv1(x)))))
        # x = self.do2(self.maxpool2(self.activ2(self.bn2(self.conv2(x)))))
        # x = self.do3(self.activ3(self.bn3(self.conv3(x))))
        x = self.maxpool1(self.activ1(self.conv1(x)))
        x = self.maxpool2(self.activ2(self.conv2(x)))
        x = self.maxpool3(self.activ3(self.conv3(x)))
        x = self.maxpool4(self.activ4(self.conv4(x)))
        x = self.maxpool5(self.activ5(self.conv5(x)))
        x = self.activ6(self.conv6(x))

        x = torch.flatten(x , start_dim = 1)

        x = self.activ4(self.fc1(x))
        return self.fc2(x)

In [ ]:
model=FlowerModel(3,5).to(device=device)

In [ ]:
model

In [ ]:
summary(model,input_size=(3,IMG_HEIGHT,IMG_WIDTH),device=device.type)